# 05 Gemini Cleanup and Evaluation

This notebook demonstrates using the Google Gemini API as a late-stage post-OCR correction step.
It compares CER/WER before and after cleanup, showing the delta contributed by the LLM correction.

The Gemini prompt is specifically tuned for 17th-century printed Spanish text:
it corrects obvious OCR mistakes without modernizing authentic period spellings.


In [ ]:
import os, sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Set your API key here or export GEMINI_API_KEY in your shell
os.environ.setdefault('GEMINI_API_KEY', 'YOUR_GEMINI_API_KEY')

from src.postprocess_llm import GeminiCleaner, RuleBasedCleaner
from src.utils import CleanupConfig, read_jsonl
from src.evaluate import compute_cer, compute_wer


## 1. Test the Gemini connection


In [ ]:
config = CleanupConfig(backend='gemini', gemini_model='models/gemini-flash-lite-latest')
gemini = GeminiCleaner(config)
print(f'Gemini available: {gemini.is_available}')

if gemini.is_available:
    test_text = 'Io, P. GARCIA de Ia COMP. de JESUS'
    cleaned, backend = gemini.clean(test_text)
    print(f'Raw:     {test_text}')
    print(f'Cleaned: {cleaned}')
    print(f'Backend: {backend}')


## 2. Compare rule-based vs Gemini cleanup on baseline predictions

Load the predictions file written by run_baseline.py and re-run both cleaners.


In [ ]:
PREDICTIONS_FILE = ROOT / 'data' / 'predictions' / 'baseline_predictions.jsonl'
GROUND_TRUTH_FILE = ROOT / 'data' / 'ground_truth' / 'ground_truth.jsonl'

if not PREDICTIONS_FILE.exists():
    print('No predictions file. Run run_baseline.py first.')
else:
    predictions = read_jsonl(PREDICTIONS_FILE)
    ground_truth = {r['page_id']: r['text'] for r in read_jsonl(GROUND_TRUTH_FILE)}
    rule_cleaner = RuleBasedCleaner(config)
    
    results = []
    for pred in predictions:
        page_id = pred['page_id']
        if page_id not in ground_truth:
            continue
        ref = ground_truth[page_id]
        raw = pred.get('raw_ocr', '')
        rule_cleaned = rule_cleaner.clean(raw)
        
        if gemini.is_available:
            gemini_cleaned, _ = gemini.clean(raw)
        else:
            gemini_cleaned = rule_cleaned
        
        results.append({
            'page_id': page_id,
            'raw_cer': compute_cer(ref, raw),
            'rule_cer': compute_cer(ref, rule_cleaned),
            'gemini_cer': compute_cer(ref, gemini_cleaned),
            'raw_wer': compute_wer(ref, raw),
            'rule_wer': compute_wer(ref, rule_cleaned),
            'gemini_wer': compute_wer(ref, gemini_cleaned),
        })
        print(f"{page_id}: raw_CER={results[-1]['raw_cer']:.4f}  rule_CER={results[-1]['rule_cer']:.4f}  gemini_CER={results[-1]['gemini_cer']:.4f}")

    if results:
        avg = lambda key: sum(r[key] for r in results) / len(results)
        print(f"\nAverage CER — Raw: {avg('raw_cer'):.4f}  Rule: {avg('rule_cer'):.4f}  Gemini: {avg('gemini_cer'):.4f}")
        print(f"Average WER — Raw: {avg('raw_wer'):.4f}  Rule: {avg('rule_wer'):.4f}  Gemini: {avg('gemini_wer'):.4f}")


## 3. Side-by-side example

Pick a single page and display the raw OCR, rule-based cleanup, and Gemini cleanup side by side.


In [ ]:
if results:
    # Use the page with the highest raw CER for a dramatic comparison
    worst = max(results, key=lambda r: r['raw_cer'])
    pred = next(p for p in predictions if p['page_id'] == worst['page_id'])
    ref = ground_truth[worst['page_id']]
    raw = pred.get('raw_ocr', '')

    print(f"=== Page: {worst['page_id']} ===")
    print(f"Raw CER {worst['raw_cer']:.4f}  ->  Gemini CER {worst['gemini_cer']:.4f}")
    print()
    print('--- GROUND TRUTH (first 400 chars) ---')
    print(ref[:400])
    print()
    print('--- RAW OCR ---')
    print(raw[:400])
    print()
    gemini_cleaned, _ = gemini.clean(raw) if gemini.is_available else (rule_cleaner.clean(raw), 'rule')
    print('--- GEMINI CLEANED ---')
    print(gemini_cleaned[:400])
